In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import kmeans as kms

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

| Название признака | Описание | Тип данных |
| :--- | :--- | :--- |
| **CUST\_ID** | Идентификатор владельца кредитной карты. | Категориальный (ID) |
| **BALANCE** | Остаток средств на счете для совершения покупок. | Числовой, Непрерывный |
| **BALANCE\_FREQUENCY** | Частота обновления баланса. Оценка от 0 до 1 (1 = часто обновляется). | Числовой, Непрерывный |
| **PURCHASES** | Общая сумма покупок, совершенных со счета. | Числовой, Непрерывный |
| **ONEOFF\_PURCHASES** | Максимальная сумма покупки, совершенной за один раз. | Числовой, Непрерывный |
| **INSTALLMENTS\_PURCHASES** | Сумма покупок, совершенных в рассрочку. | Числовой, Непрерывный |
| **CASH\_ADVANCE** | Сумма полученного денежного аванса (наличными). | Числовой, Непрерывный |
| **PURCHASES\_FREQUENCY** | Частота совершения покупок (от 0 до 1). | Числовой, Непрерывный |
| **ONEOFFPURCHASESFREQUENCY** | Частота совершения покупок за один раз (от 0 до 1). | Числовой, Непрерывный |
| **PURCHASESINSTALLMENTSFREQUENCY** | Частота совершения покупок в рассрочку (от 0 до 1). | Числовой, Непрерывный |
| **CASHADVANCEFREQUENCY** | Частота получения денежного аванса (от 0 до 1). | Числовой, Непрерывный |
| **CASHADVANCETRX** | Количество транзакций с получением денежного аванса. | Числовой, Дискретный |
| **PURCHASES\_TRX** | Количество совершенных транзакций по покупкам. | Числовой, Дискретный |
| **CREDIT\_LIMIT** | Максимальный кредитный лимит пользователя. | Числовой, Непрерывный |
| **PAYMENTS** | Сумма платежей, внесенных пользователем. | Числовой, Непрерывный |
| **MINIMUM\_PAYMENTS** | Сумма минимальных платежей, внесенных пользователем. | Числовой, Непрерывный |
| **PRCFULLPAYMENT** | Процент полного погашения, внесенного пользователем. | Числовой, Непрерывный |
| **TENURE** | Срок обслуживания кредитной карты (в месяцах или годах, исходя из контекста данных). | Числовой, Дискретный |

In [ ]:
df = pd.read_csv('dataset.csv')
df_raw = df.copy()

df.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


# Применяем KMeans

In [ ]:
X = df[['BALANCE', 'PURCHASES']]
CLUSTERS_COUNT = 4

kmeans : kms.KMeans = kms.KMeansImpl(
    X=X.to_numpy(),
    clusters_count=CLUSTERS_COUNT,
    stopper= kms.KMeansIterationStoperItersCount(10),
)

kmeans.fit()

clusters = [kmeans.predict(x) for x in X.to_numpy()]

# print(clusters)

X['СLUSTER'] = clusters

fig = px.scatter(
    X,
    x='BALANCE',
    y='PURCHASES',
    color='СLUSTER',
    title='Распределение данных по BALANCE и PURCHASES',
    labels={'BALANCE': 'Баланс', 'PURCHASES': 'Покупки'},
    opacity=0.6
)

fig.show()

/var/folders/y7/3khx9vbx74z9_0bd7nswgf080000gn/T/ipykernel_73156/4108520604.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['СLUSTER'] = clusters


# Метод локтя

In [ ]:
X = df[['BALANCE', 'PURCHASES']]

elbow_trainer: kms.ElbowTrainer = kms.ElbowTrainerImpl(
    X=X.to_numpy(),
    ks=range(1, 11),
    km_stopper= kms.KMeansIterationStoperItersCount(10),
)

elbow_results = elbow_trainer.train()


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=[res['k'] for res in elbow_results],
        y=[res['inertia'] for res in elbow_results],
        mode='lines+markers',
        name='Inertia vs K',
    )
)

fig.show()